# ML Competition 1 — Student Notebook
### ML Bootcamp 2026 · University of Toronto · Week 1

This is a team-based binary classification challenge. Your team receives labeled data from a bank marketing campaign, builds a model, and uses that model to predict whether a client will subscribe to a term deposit. A simple working example is provided in this notebook, and you should improve it.

## 1. The data and the task

The data comes from the direct marketing campaigns (phone calls) of a Portuguese bank. Each row describes one contact with a client.

**The task is binary classification.** This is a classification problem, not a regression problem: the model predicts a class label, not a continuous number. For each client, your model predicts one of two classes:

- **1** — the client subscribed to a term deposit ("yes"),
- **0** — the client did not subscribe ("no").

The positive class (a subscription, label `1`) is the minority: only about **11%** of clients subscribed. Because the two classes are imbalanced, the competition is scored with the **F1 score**, not accuracy.

The complete dataset contains about **41,000 client records** and **20 input features**: **10 numeric** (for example, age and the number of contacts) and **10 categorical** (for example, job, marital status, and education). It has been divided into the three files below.

You will receive three files. **All three files contain missing values**, so your code must be able to handle missing data. The files contain different rows taken from the same dataset.

| File | Contents | Labels | When you receive it |
|---|---|---|---|
| `train.csv` | real data with missing values | **Yes** | at the start |
| `test_public.csv` | different rows, with missing values | **Yes** | at the start |
| `test_final.csv` | different rows, with missing values **and a small shift** | Yes (released with the file) | **at the end (one attempt)** |

The file `test_final.csv` is similar to `test_public.csv`, but a small amount of noise has been added. Because `test_public.csv` already contains missing values, any code that runs correctly on it will also run on `test_final.csv`.

## 2. Scoring and the live leaderboard

The competition is scored using the **F1 score of the positive class** (label `1`).

The file `test_public.csv` includes labels, so you can compute your own F1 score and report it. These scores are shown on a **live leaderboard** that is updated during the afternoon. The leaderboard is for feedback only and does not determine the winner.

At the end of the session, `test_final.csv` is released one time. You run your existing `predict` function on this file, and your **F1 score on `test_final.csv` determines the final ranking**.

## 3. Suggested division of work

The following roles are suggestions for a team of four. You may divide and combine the work in the way that best suits your team.

- **Data and Preprocessing.** Exploratory data analysis; handling missing values; encoding and scaling within the pipeline.
- **Feature Engineering.** Creating and selecting features; analyzing correlations and feature importance; optionally creating features using PCA or K-means from Day 3.
- **Modeling.** Training the candidate models (logistic regression, KNN, decision trees, ensembles) and making sure each model runs correctly.
- **Validation and Tuning.** Choosing a validation method (a train/validation split or cross-validation); performing hyperparameter tuning; and comparing the models to decide which one to submit.

## 4. Rules

- **Only classical machine learning methods from Days 1–3 may be used:** linear and logistic regression, KNN, decision trees, and ensembles (random forests, bagging, boosting). Deep learning is not permitted.
- **No external data may be used.** Train your model only on `train.csv`.
- **The `predict` function must not re-fit** any imputer, scaler, or encoder on the test set, and must not use the label column.
- *(Possible time limit, to be confirmed:* `train` and `predict` should finish within approximately 2 minutes on a Colab CPU.*)*

## 5. The code interface you must implement

Your submission must define exactly two functions with the following signatures:

```python
def train(train_df):    # -> checkpoint
def predict(checkpoint, test_df):    # -> predictions
```

- **`train`** fits all components (imputation, encoding, scaling, feature engineering, and the model) and returns a single **checkpoint** that contains every fitted component.
- **`predict`** receives this checkpoint and a test dataframe, and returns one prediction for each row. It applies the components that were already fitted during training. It must not re-fit anything on the test set, must not read the label column, and must be able to handle missing values.

A fitted scikit-learn `Pipeline` (impute → encode → scale → model) is a convenient way to meet these requirements. The cells below show an example.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

In [ ]:
# --- These files will be provided to you ---
TRAIN_PATH       = "https://raw.githubusercontent.com/IIB-Lab/AI-ML-bootcamp-tutorials/refs/heads/main/06-competitions/ML%20Competition%201/train.csv"        # provided at the start (includes labels)
TEST_PUBLIC_PATH = "https://raw.githubusercontent.com/IIB-Lab/AI-ML-bootcamp-tutorials/refs/heads/main/06-competitions/ML%20Competition%201/test_public.csv"  # provided at the start (includes labels)
# change test_public.csv to test_final.csv  ->  provided once at the end, in the same format as test_public.csv

TARGET = "y"   # label column: 1 = subscribed, 0 = did not subscribe

train_df       = pd.read_csv(TRAIN_PATH)
test_public_df = pd.read_csv(TEST_PUBLIC_PATH)

print("train:", train_df.shape, " | test_public:", test_public_df.shape)
train_df.head()

### Minimal working example (you should replace this)

The pipeline below automatically detects numeric and categorical columns, so it runs without changing column names. It imputes missing values, encodes categorical features, scales the data, and fits a logistic regression model. This is only a starting baseline. You should improve the preprocessing, create new features, and try other models such as KNN, decision trees, or ensembles.

In [ ]:
def build_pipeline():
    """A minimal example pipeline. You may replace or extend it.
    It automatically detects numeric and categorical columns, so it runs on any dataset."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    pre = ColumnTransformer([
        ("num", numeric,     make_column_selector(dtype_include=np.number)),
        ("cat", categorical, make_column_selector(dtype_include=object)),
    ])
    return Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000))])


def train(train_df):
    """Fit all components here. Return one checkpoint that contains every fitted component."""
    X = train_df.drop(columns=[TARGET])
    y = train_df[TARGET]

    pipe = build_pipeline()
    pipe.fit(X, y)

    checkpoint = {"pipeline": pipe, "features": list(X.columns)}
    return checkpoint


def predict(checkpoint, test_df):
    """Apply the checkpoint that was already fitted during training.
    Do not re-fit anything on test_df. Do not use the label column.
    The pipeline handles missing values."""
    X = test_df.drop(columns=[TARGET], errors="ignore")   # do not use the labels
    X = X[checkpoint["features"]]                          # use the same columns and order as in training
    return checkpoint["pipeline"].predict(X)

### Check your code and compute your score

This cell checks that your `train` and `predict` functions run correctly on the public test set, and prints your **F1 score** for the leaderboard (accuracy is also shown, for reference only). If this cell runs without errors, your code will also run on the final test file.

In [ ]:
ckpt  = train(train_df)
preds = predict(ckpt, test_public_df)
assert len(preds) == len(test_public_df), "You must return one prediction for each row."

f1  = f1_score(test_public_df[TARGET], preds)
acc = accuracy_score(test_public_df[TARGET], preds)
print(f"Public F1 (leaderboard metric): {f1:.4f}")
print(f"Public accuracy (reference)   : {acc:.4f}")

## 6. Ideas to explore (optional)

These are optional directions. You are not required to use any of them, but they connect directly to the methods from Days 1–3.

- **Dimensionality test.** Compare model performance using all features against a smaller set of components created with **PCA**. Fewer dimensions can reduce overfitting and speed up training.
- **Missing values.** Dropping rows or columns is only one option, and it throws away data. Consider **imputing** missing values instead — for example, filling with the mean, median, or most frequent value, or using KNN-based imputation — which usually keeps more information.
- **New features from clustering.** Run **K-means** (or another clustering method) on the data and use the resulting cluster label as a new categorical feature.
- **Feature engineering.** Create new features by combining or transforming existing ones, and check whether they improve the score.
- **Model choice and tuning.** Compare several models (logistic regression, KNN, decision trees, ensembles) and tune their hyperparameters.
- **Class imbalance.** Because the positive class is a minority (about 11%), consider techniques such as class weighting, resampling, or adjusting the decision threshold to improve the F1 score.